# Step E — DL Top-1% Anchor, Final SVD Polarity & Comparison with Raw DL
Re-implements `E_Po_all_15OBS.m`, replacing the manual anchor (step D) with  
**DL picks whose confidence is in the top 1% for that station**.  

For each station:  
1. Find the top-1% confidence threshold → printed for reference  
2. Compute `WTcatpol = Σ(DL_pol × U_val) / Σ|U_val|` over anchor events  
3. Assign `SVD_final = sign(U × sign(WTcatpol))` if `|U| > threshold`, else 0  
4. Compare SVD_final vs raw DL polarity event-by-event  
5. Plot top-20 waveforms + polarity markers per station  
6. Save to `03-output-graphics/cc_svd_validation/`

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import h5py
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

DATA   = Path('/Users/mczhang/Documents/GitHub/FM7/02-data')
OUT_C  = DATA / 'C_SVD_py'
OUT_E  = DATA / 'E_Po_py'; OUT_E.mkdir(exist_ok=True)
WAVE_H5   = DATA / 'A_ID'  / 'waves_Y2_ge8.h5'
DL_H5     = DATA / 'E_Po'  / 'dl_polarity_Y2_ge8.h5'
FIG_DIR   = Path('/Users/mczhang/Documents/GitHub/FM7/03-output-graphics/cc_svd_validation')
FIG_DIR.mkdir(parents=True, exist_ok=True)
YEAR = 'Y2'
FS   = 200
T    = np.arange(264) / FS - 0.32   # time axis relative to P pick

DL_CONF_MIN = 0.8   # minimum DL confidence for agreement comparison
print('=== Top-3% DL Confidence Thresholds (for reference) ===')


=== Top-3% DL Confidence Thresholds (for reference) ===


In [2]:
# Pre-load DL data
with h5py.File(DL_H5) as f:
    DL_EIDS = f['event_ids'][:]
    DL_STAS = sorted(f['polarity'].keys())
    DL_POL  = {s: f[f'polarity/{s}'][:].astype(int)   for s in DL_STAS}
    DL_CONF = {s: f[f'confidence/{s}'][:].astype(float) for s in DL_STAS}

with h5py.File(WAVE_H5) as f:
    WAVE_EIDS = f['event_ids'][:]
    WAVE_STAS = sorted(f['waveforms'].keys())

# Build eid → index maps
dl_eid2idx   = {int(e): i for i, e in enumerate(DL_EIDS)}
wave_eid2idx = {int(e): i for i, e in enumerate(WAVE_EIDS)}


In [3]:
# ── Main processing ─────────────────────────────────────────
SUMMARY = []
N_TOTAL_EVENTS = len(DL_EIDS)   # total events in dataset

for C_path in sorted(OUT_C.glob(f'C_{YEAR}_*.npy')):
    sta = C_path.stem.replace(f'C_{YEAR}_', '')
    svd_result = np.load(C_path)   # (N, 6): eid, u_val, svd_pol, agree, disagree, thr_log

    if sta not in DL_POL:
        print(f'{sta}: no DL data, skip')
        continue

    # Align DL data to SVD events
    svd_eids = svd_result[:, 0].astype(int)
    N = len(svd_eids)

    dl_pol  = np.zeros(N, dtype=int)
    dl_conf = np.zeros(N)
    for k, eid in enumerate(svd_eids):
        if eid in dl_eid2idx:
            idx = dl_eid2idx[eid]
            dl_pol[k]  = DL_POL[sta][idx]
            dl_conf[k] = DL_CONF[sta][idx]

    # ── Top-1% confidence threshold ────────────────────────
    has_pick = dl_pol != 0
    if has_pick.sum() < 5:
        print(f'{sta}: too few DL picks ({has_pick.sum()}), skip')
        continue
    conf_vals = dl_conf[has_pick]
    top1_thr  = np.percentile(conf_vals, 97)
    n_top3    = (conf_vals >= top1_thr).sum()
    print(f'{sta:5s}: top-3% conf threshold = {top1_thr:.6f}  (n_anchors = {n_top3})')

    # ── WTcatpol: anchor with top-1% DL picks ──────────────
    anchor_mask = has_pick & (dl_conf >= top1_thr)
    if anchor_mask.sum() == 0:
        print(f'  {sta}: no anchor events above top-1% threshold, skip')
        continue

    u_vals    = svd_result[:, 1]
    thr_log   = svd_result[0, 5]
    n_matrix  = len(svd_eids)   # events in SVD matrix
    WTcatpol = (np.sum(dl_pol[anchor_mask] * u_vals[anchor_mask]) /
                np.sum(np.abs(u_vals[anchor_mask])))

    # ── Final SVD polarity ──────────────────────────────────
    thr_val = 10 ** thr_log
    svd_final = np.sign(u_vals * np.sign(WTcatpol))
    svd_final[np.abs(u_vals) < thr_val] = 0   # below threshold → no pick

    # ── Comparison (all DL + high-conf DL >= 0.8) ──────────
    dl_hq     = has_pick & (dl_conf >= DL_CONF_MIN)
    both_all  = (svd_final != 0) & has_pick
    both_hq   = (svd_final != 0) & dl_hq
    agree_all = both_all & (svd_final == dl_pol)
    agree_hq  = both_hq  & (svd_final == dl_pol)
    n_svd     = (svd_final != 0).sum()
    n_dl      = has_pick.sum()
    n_dl_hq   = dl_hq.sum()
    n_both    = both_all.sum()
    n_both_hq = both_hq.sum()
    n_agree   = agree_all.sum()
    n_agree_hq= agree_hq.sum()
    agr_rate  = n_agree    / max(n_both,    1)
    agr_hq    = n_agree_hq / max(n_both_hq, 1)

    cc_det_pct  = round(n_svd / N_TOTAL_EVENTS * 100, 1)
    pct_in_svd   = round(n_svd / max(n_matrix, 1) * 100, 1)
    SUMMARY.append(dict(
        station=sta, WTcatpol=round(WTcatpol, 4),
        top3_thr=round(top1_thr, 6), n_anchors=n_top3,
        n_matrix=n_matrix, n_svd=n_svd,
        pct_in_svd=pct_in_svd, cc_det_pct=cc_det_pct,
        svd_thr=f'1e{thr_log:.0f}',
        n_dl=n_dl, n_dl_hq=n_dl_hq,
        n_both=n_both, n_agree=n_agree, agree_pct=round(agr_rate*100, 1),
        n_both_hq=n_both_hq, n_agree_hq=n_agree_hq, agree_hq_pct=round(agr_hq*100, 1)
    ))

    # Save per-station results
    out_arr = np.column_stack([svd_eids, u_vals, svd_final, dl_pol, dl_conf,
                               np.full(N, WTcatpol), np.full(N, top1_thr)])
    np.save(OUT_E / f'E_{YEAR}_{sta}.npy', out_arr)

print()
df_sum = pd.DataFrame(SUMMARY)
if len(df_sum):
    cols = ['station','n_matrix','n_svd','pct_in_svd','cc_det_pct','svd_thr',
            'n_dl','n_agree','agree_pct',
            'n_dl_hq','n_both_hq','n_agree_hq','agree_hq_pct']
    print('Comparison: all DL picks vs DL conf >= 0.8')
    print(df_sum[cols].to_string(index=False))
    df_sum.to_csv(OUT_E / f'E_{YEAR}_summary.csv', index=False)


01B  : top-3% conf threshold = 0.999931  (n_anchors = 85)
02B  : top-3% conf threshold = 0.999978  (n_anchors = 151)
03B  : top-3% conf threshold = 0.999986  (n_anchors = 171)
04B  : top-3% conf threshold = 0.999995  (n_anchors = 138)
05B  : top-3% conf threshold = 0.999968  (n_anchors = 207)
06B  : top-3% conf threshold = 0.999941  (n_anchors = 258)
07B  : top-3% conf threshold = 0.999990  (n_anchors = 382)
08B  : top-3% conf threshold = 0.999922  (n_anchors = 333)
09B  : top-3% conf threshold = 0.999997  (n_anchors = 204)
10B  : top-3% conf threshold = 0.999989  (n_anchors = 439)
11B  : top-3% conf threshold = 0.999985  (n_anchors = 348)
12B  : top-3% conf threshold = 0.999984  (n_anchors = 199)
13B  : top-3% conf threshold = 0.999262  (n_anchors = 132)
14B  : top-3% conf threshold = 0.999978  (n_anchors = 134)
AS1  : top-3% conf threshold = 0.999991  (n_anchors = 276)


AS2  : top-3% conf threshold = 0.999982  (n_anchors = 238)
CC1  : top-3% conf threshold = 0.999989  (n_anchors = 468)
EC1  : top-3% conf threshold = 0.999999  (n_anchors = 426)
EC2  : top-3% conf threshold = 0.999963  (n_anchors = 409)


EC3  : top-3% conf threshold = 0.999993  (n_anchors = 351)
ID1  : top-3% conf threshold = 0.999991  (n_anchors = 130)

Comparison: all DL picks vs DL conf >= 0.8
station  n_matrix  n_svd  pct_in_svd  cc_det_pct svd_thr  n_dl  n_agree  agree_pct  n_dl_hq  n_both_hq  n_agree_hq  agree_hq_pct
    01B      2819   2143        76.0        10.4    1e-7  2819     1328       62.0     1960       1561        1016          65.1
    02B      5002   3846        76.9        18.6    1e-8  5002     2642       68.7     3779       3039        2195          72.2
    03B      5684   5113        90.0        24.8    1e-8  5684     2807       54.9     4458       4068        2260          55.6
    04B      4585   4016        87.6        19.5    1e-9  4585     1903       47.4     3367       2976        1401          47.1
    05B      6855   6178        90.1        29.9    1e-9  6855     4009       64.9     4844       4455        3093          69.4
    06B      8574   6797        79.3        32.9   1e-10  8574  

In [4]:
# ── Top-20 waveform + polarity plots ────────────────────────
with h5py.File(WAVE_H5) as f:
    WAVES_ALL = {s: f[f'waveforms/{s}'][:] for s in WAVE_STAS}

POL_COLORS = {1: '#d62728', -1: '#2166ac', 0: 'gray'}
MARKER_S   = {1: '^', -1: 'v', 0: 'o'}
MARKER_SZ  = 120

for E_path in sorted(OUT_E.glob(f'E_{YEAR}_*.npy')):
    sta = E_path.stem.replace(f'E_{YEAR}_', '')
    arr = np.load(E_path)
    # cols: eid, u_val, svd_final, dl_pol, dl_conf, WTcatpol, top1_thr
    eids_s    = arr[:, 0].astype(int)
    u_vals    = arr[:, 1]
    svd_final = arr[:, 2].astype(int)
    dl_pol    = arr[:, 3].astype(int)
    dl_conf   = arr[:, 4]

    # Select 20 random anchor events (top-1% DL conf — these set WTcatpol)
    top1_thr  = arr[0, 6]
    anchor_idx = np.where(dl_conf >= top1_thr)[0]
    rng = np.random.default_rng(seed=42)
    if len(anchor_idx) > 20:
        top_idx = rng.choice(anchor_idx, size=20, replace=False)
        top_idx = top_idx[np.argsort(-np.abs(u_vals[top_idx]))]  # sort by |u| for display
    else:
        top_idx = anchor_idx[np.argsort(-np.abs(u_vals[anchor_idx]))]
    n_show = len(top_idx)

    fig, axes = plt.subplots(n_show, 1, figsize=(12, 2*n_show), facecolor='white')
    if n_show == 1:
        axes = [axes]
    fig.suptitle(f'Station {sta} | {n_show} random anchor waveforms (top-1% DL conf) | Y1'
                 f' -- Red=Up Blue=Down (SVD filled / DL open)',
                 fontsize=11, fontweight='bold', y=1.001)

    for row, k in enumerate(top_idx):
        ax = axes[row]
        eid = eids_s[k]

        # Load waveform
        wave = None
        if sta in WAVES_ALL and eid in wave_eid2idx:
            w = WAVES_ALL[sta][wave_eid2idx[eid]]
            if np.isfinite(w).all():
                wave = w

        if wave is not None:
            peak = np.max(np.abs(wave)) or 1.0
            ax.plot(T, wave / peak, color='#555555', lw=0.8, zorder=1)
        else:
            ax.text(0, 0, 'no waveform', ha='center', va='center',
                    color='gray', fontsize=8)

        # Shade CC windows
        ax.axvspan(T[44], T[103], color='lightyellow', alpha=0.5, zorder=0)
        ax.axvspan(T[44], T[143], color='lightyellow', alpha=0.25, zorder=0)
        ax.axvline(0, color='k', lw=0.7, ls='--', zorder=2)

        # SVD polarity marker (filled)
        sp = svd_final[k]
        if sp != 0:
            ax.scatter([0.06], [0.6 * sp], s=MARKER_SZ,
                       marker=MARKER_S[sp], color=POL_COLORS[sp],
                       zorder=5, label=f'SVD={sp:+d}')

        # DL polarity marker (open)
        dp = dl_pol[k]
        if dp != 0:
            ax.scatter([0.12], [0.6 * dp], s=MARKER_SZ,
                       marker=MARKER_S[dp], facecolors='none',
                       edgecolors=POL_COLORS[dp], linewidths=1.5,
                       zorder=5, label=f'DL={dp:+d}(c={dl_conf[k]:.2f})')

        # Agree / disagree label
        match = (sp == dp and sp != 0)
        lbl = '✓' if match else ('✗' if sp != 0 and dp != 0 else '?')
        lbl_col = 'green' if match else 'red'
        ax.set_xlim(-0.15, 0.6)
        ax.set_ylim(-1.4, 1.4)
        ax.set_yticks([])
        title = (f'eid={eid}  |U|={abs(u_vals[k]):.4f}  '
                 f'SVD={sp:+d}  DL={dp:+d}(c={dl_conf[k]:.2f})  {lbl}')
        ax.set_title(title, fontsize=7.5, color=lbl_col if sp != 0 and dp != 0 else '#333333',
                     pad=1)
        ax.tick_params(labelsize=6)
        if row == n_show - 1:
            ax.set_xlabel('Time relative to P pick (s)', fontsize=8)

    plt.tight_layout()
    out_fig = FIG_DIR / f'anchor20_{YEAR}_{sta}.png'
    fig.savefig(out_fig, dpi=100, bbox_inches='tight')
    plt.close(fig)
    print(f'  {sta}: {n_show} anchors plotted → {out_fig.name}')

print('\nStep E done. Figures in:', FIG_DIR)

  01B: 20 anchors plotted → anchor20_Y2_01B.png


  02B: 20 anchors plotted → anchor20_Y2_02B.png


  03B: 20 anchors plotted → anchor20_Y2_03B.png


  04B: 20 anchors plotted → anchor20_Y2_04B.png


  05B: 20 anchors plotted → anchor20_Y2_05B.png


...

  CC1: 20 anchors plotted → anchor20_Y2_CC1.png


  EC1: 20 anchors plotted → anchor20_Y2_EC1.png


  EC2: 20 anchors plotted → anchor20_Y2_EC2.png


  EC3: 20 anchors plotted → anchor20_Y2_EC3.png


  ID1: 20 anchors plotted → anchor20_Y2_ID1.png

Step E done. Figures in: /Users/mczhang/Documents/GitHub/FM7/03-output-graphics/cc_svd_validation


In [5]:
# ── Summary plot: agreement rate by station ──────────────────
df = pd.read_csv(OUT_E / f'E_{YEAR}_summary.csv')
df = df.sort_values('agree_pct', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
colors = ['#2196f3' if 'A' in s or 'B' in s or any(c.isdigit() for c in s[:3])
          else '#4caf50' for s in df['station']]
bars = ax.barh(df['station'], df['agree_pct'], color=colors, alpha=0.85, edgecolor='none')
ax.axvline(90, color='gray', ls='--', lw=1, label='90%')
ax.axvline(95, color='red',  ls='--', lw=1, label='95%')
for bar, row in zip(bars, df.itertuples()):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{row.agree_pct:.1f}%  (n={row.n_both})',
            va='center', fontsize=8)
ax.set_xlabel('SVD–DL Agreement Rate (%)', fontsize=10)
ax.set_title(f'SVD-CC vs Raw DL Polarity Agreement by Station  |  Y1', fontsize=11)
ax.legend(fontsize=9)
ax.set_xlim(0, 110)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
out_sum = FIG_DIR / f'agreement_summary_{YEAR}.png'
fig.savefig(out_sum, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Summary figure → {out_sum}')
df


Summary figure → /Users/mczhang/Documents/GitHub/FM7/03-output-graphics/cc_svd_validation/agreement_summary_Y2.png


,station,WTcatpol,top3_thr,n_anchors,n_matrix,n_svd,pct_in_svd,cc_det_pct,svd_thr,n_dl,n_dl_hq,n_both,n_agree,agree_pct,n_both_hq,n_agree_hq,agree_hq_pct
3,04B,-0.9892,0.999995,138,4585,4016,87.6,19.5,1.000000e-09,4585,3367,4016,1903,47.4,2976,1401,47.1
12,13B,0.5582,0.999262,132,4374,3953,90.4,19.2,1.000000e-06,4374,2669,3953,2061,52.1,2415,1273,52.7
2,03B,-0.8280,0.999986,171,5684,5113,90.0,24.8,1.000000e-08,5684,4458,5113,2807,54.9,4068,2260,55.6
18,EC2,-0.0277,0.999963,409,13544,12191,90.0,59.1,1.000000e-11,13544,9264,12191,6706,55.0,8330,4902,58.8
8,09B,0.1651,0.999997,204,6717,5974,88.9,28.9,1.000000e-07,6717,4788,5974,3333,55.8,4317,2445,56.6
15,AS2,0.7629,0.999982,238,7903,6826,86.4,33.1,1.000000e-09,7903,5653,6826,3851,56.4,4950,2866,57.9
11,12B,-0.3597,0.999984,199,6580,5045,76.7,24.4,1.000000e-08,6580,4579,5045,2874,57.0,3608,2112,58.5
16,CC1,0.3730,0.999989,468,15345,14379,93.7,69.7,1.000000e-09,15345,10830,14379,8221,57.2,10105,6345,62.8
0,01B,-0.9955,0.999931,85,2819,2143,76.0,10.4,1.000000e-07,2819,1960,2143,1328,62.0,1561,1016,65.1
4,05B,-0.9699,0.999968,207,6855,6178,90.1,29.9,1.000000e-09,6855,4844,6178,4009,64.9,4455,3093,69.4


In [6]:
# ── Print summary table as PDF ───────────────────────────────────────────────
import matplotlib.backends.backend_pdf as pdf_backend
from matplotlib.patches import FancyBboxPatch

df = pd.read_csv(OUT_E / f'E_{YEAR}_summary.csv')
cols_show = ['station','n_matrix','n_svd','pct_in_svd','cc_det_pct','svd_thr',
             'n_dl','n_agree','agree_pct',
             'n_dl_hq','n_both_hq','n_agree_hq','agree_hq_pct']
col_labels = ['Station','N in SVD','N polarity','% in SVD','CC det%','SVD thr',
              '#DL all','#Agree all','Agree%(all)',
              '#DL>=0.8','#Both>=0.8','#Agree>=0.8','Agree%(>=0.8)']

data = df[cols_show].values.tolist()

fig, ax = plt.subplots(figsize=(18, 0.45 * len(data) + 1.5))
ax.axis('off')

tbl = ax.table(
    cellText=data,
    colLabels=col_labels,
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.6)

# Header style
for j in range(len(col_labels)):
    tbl[(0, j)].set_facecolor('#2c3e50')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

# Row alternating colours; highlight low agree% in red, high in green
for i, row in enumerate(data, start=1):
    agree = float(row[-1])   # agree_hq_pct
    wt    = abs(float(row[1])) if row[1] != '' else 0   # top3_thr is float
    for j in range(len(col_labels)):
        cell = tbl[(i, j)]
        cell.set_facecolor('#f2f2f2' if i % 2 == 0 else 'white')
    # Agree% cell colour
    agree_cell = tbl[(i, len(col_labels) - 1)]   # last col = Agree%(>=0.8)
    if agree >= 75:
        agree_cell.set_facecolor('#c8e6c9')
    elif agree < 50:
        agree_cell.set_facecolor('#ffcdd2')
    else:
        agree_cell.set_facecolor('#fff9c4')

ax.set_title(f'CC-SVD vs DL Polarity Comparison — Y1 (N=3105 events)\n'
             f'Top-3% DL anchor | left=all DL  right=DL conf>=0.8',
             fontsize=11, fontweight='bold', pad=12)

plt.tight_layout()
out_pdf = FIG_DIR / f'E_{YEAR}_summary_table.pdf'
out_png = FIG_DIR / f'E_{YEAR}_summary_table.png'
fig.savefig(out_pdf, dpi=150, bbox_inches='tight')
fig.savefig(out_png, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved → {out_pdf}')
print(f'Saved → {out_png}')
df[cols_show]


Saved → /Users/mczhang/Documents/GitHub/FM7/03-output-graphics/cc_svd_validation/E_Y2_summary_table.pdf
Saved → /Users/mczhang/Documents/GitHub/FM7/03-output-graphics/cc_svd_validation/E_Y2_summary_table.png


,station,n_matrix,n_svd,pct_in_svd,cc_det_pct,svd_thr,n_dl,n_agree,agree_pct,n_dl_hq,n_both_hq,n_agree_hq,agree_hq_pct
0,01B,2819,2143,76.0,10.4,1.000000e-07,2819,1328,62.0,1960,1561,1016,65.1
1,02B,5002,3846,76.9,18.6,1.000000e-08,5002,2642,68.7,3779,3039,2195,72.2
2,03B,5684,5113,90.0,24.8,1.000000e-08,5684,2807,54.9,4458,4068,2260,55.6
3,04B,4585,4016,87.6,19.5,1.000000e-09,4585,1903,47.4,3367,2976,1401,47.1
4,05B,6855,6178,90.1,29.9,1.000000e-09,6855,4009,64.9,4844,4455,3093,69.4
5,06B,8574,6797,79.3,32.9,1.000000e-10,8574,4685,68.9,6067,4866,3550,73.0
6,07B,12724,10890,85.6,52.8,1.000000e-07,12724,8740,80.3,10134,8922,7609,85.3
7,08B,11089,8982,81.0,43.5,1.000000e-08,11089,6606,73.5,8299,6919,5377,77.7
8,09B,6717,5974,88.9,28.9,1.000000e-07,6717,3333,55.8,4788,4317,2445,56.6
9,10B,14545,12764,87.8,61.9,1.000000e-06,14545,9170,71.8,11580,10499,7913,75.4
